# User-User Collaborative Filtering

## Learning Objectives

1. **Describe** the utility matrix and the rating prediction problem
2. **Define** Pearson correlation as a user similarity measure and explain mean-centring
3. **Derive** the weighted-average prediction formula
4. **Identify** scalability problems with user-user CF and when item-item is preferred
5. **Implement** Pearson-based user-user CF with k-nearest-neighbour selection


## Problem Statement

### The Recommendation Problem

Given a sparse **utility matrix** $R$ (rows = users, columns = items, entries = ratings), predict the missing entries. The key assumption of collaborative filtering (CF) is:

> Users who agreed in the past will tend to agree in the future.

**User-user CF:** For user $u$ and unrated item $i$, find users $v$ who rated $i$ and are most similar to $u$. Predict $\hat{r}_{ui}$ as a weighted average of their ratings.

### Challenges

- **Sparsity:** Most user-item pairs are unrated (>99% for large systems)
- **Scalability:** Computing pairwise similarity for $n = 10^8$ users is $O(n^2)$
- **Cold start:** New users have no ratings to compare against
- **User taste drift:** Preferences change over time


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="380" font-family="monospace" font-size="12">
  <rect width="820" height="380" fill="#fafafa" rx="8"/>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">User-User Collaborative Filtering</text>

  <!-- Utility matrix -->
  <text x="210" y="48" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Utility Matrix (ratings, ? = missing)</text>

  <rect x="20" y="58" width="380" height="170" rx="4" fill="white" stroke="#ccc" stroke-width="1"/>
  <!-- header -->
  <rect x="20" y="58" width="380" height="24" rx="4" fill="#ddd"/>
  <text x="80"  y="75" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">User</text>
  <text x="165" y="75" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Item A</text>
  <text x="225" y="75" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Item B</text>
  <text x="285" y="75" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Item C</text>
  <text x="345" y="75" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Item D</text>
  <!-- rows -->
  <text x="80" y="100" text-anchor="middle" fill="#4e79a7" font-size="11">Alice</text>   <text x="165" y="100" text-anchor="middle" fill="#333" font-size="11">5</text><text x="225" y="100" text-anchor="middle" fill="#333" font-size="11">4</text><text x="285" y="100" text-anchor="middle" fill="#555" font-size="11">?</text><text x="345" y="100" text-anchor="middle" fill="#333" font-size="11">2</text>
  <text x="80" y="125" text-anchor="middle" fill="#333" font-size="11">Bob</text>     <text x="165" y="125" text-anchor="middle" fill="#555" font-size="11">?</text><text x="225" y="125" text-anchor="middle" fill="#333" font-size="11">3</text><text x="285" y="125" text-anchor="middle" fill="#333" font-size="11">4</text><text x="345" y="125" text-anchor="middle" fill="#555" font-size="11">?</text>
  <text x="80" y="150" text-anchor="middle" fill="#333" font-size="11">Carol</text>   <text x="165" y="150" text-anchor="middle" fill="#333" font-size="11">4</text><text x="225" y="150" text-anchor="middle" fill="#555" font-size="11">?</text><text x="285" y="150" text-anchor="middle" fill="#333" font-size="11">5</text><text x="345" y="150" text-anchor="middle" fill="#333" font-size="11">3</text>
  <text x="80" y="175" text-anchor="middle" fill="#333" font-size="11">Dave</text>    <text x="165" y="175" text-anchor="middle" fill="#333" font-size="11">2</text><text x="225" y="175" text-anchor="middle" fill="#333" font-size="11">1</text><text x="285" y="175" text-anchor="middle" fill="#555" font-size="11">?</text><text x="345" y="175" text-anchor="middle" fill="#333" font-size="11">5</text>
  <!-- Predict Alice, Item C (highlighted) -->
  <rect x="255" y="87" width="60" height="22" rx="3" fill="#ffd700" opacity="0.6"/>
  <text x="285" y="103" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">predict</text>

  <!-- Pearson similarity -->
  <rect x="20" y="242" width="380" height="60" rx="4" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="210" y="262" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Pearson Correlation (mean-centred)</text>
  <text x="210" y="282" text-anchor="middle" fill="#333" font-size="11">sim(u,v) = Σ(rᵤᵢ−r̄ᵤ)(rᵥᵢ−r̄ᵥ) / [||rᵤ−r̄ᵤ|| · ||rᵥ−r̄ᵥ||]</text>
  <text x="210" y="296" text-anchor="middle" fill="#555" font-size="10">sum over items rated by BOTH u and v</text>

  <!-- Prediction formula -->
  <rect x="20" y="314" width="380" height="55" rx="4" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="210" y="334" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">Rating Prediction</text>
  <text x="210" y="352" text-anchor="middle" fill="#333" font-size="12">r̂ᵤᵢ = r̄ᵤ + Σᵥ sim(u,v)·(rᵥᵢ−r̄ᵥ) / Σᵥ|sim(u,v)|</text>
  <text x="210" y="366" text-anchor="middle" fill="#555" font-size="10">sum over k nearest neighbours v who rated item i</text>

  <!-- Similarity heatmap (right) -->
  <text x="610" y="48" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">User Similarity Matrix</text>
  <rect x="440" y="58" width="360" height="170" rx="4" fill="white" stroke="#ccc" stroke-width="1"/>
  <rect x="440" y="58" width="360" height="24" rx="4" fill="#ddd"/>
  <text x="500" y="75" text-anchor="middle" fill="#555" font-size="11" font-weight="bold">sim</text>
  <text x="575" y="75" text-anchor="middle" fill="#4e79a7" font-size="11">Alice</text>
  <text x="645" y="75" text-anchor="middle" fill="#333" font-size="11">Bob</text>
  <text x="715" y="75" text-anchor="middle" fill="#333" font-size="11">Carol</text>
  <text x="785" y="75" text-anchor="middle" fill="#333" font-size="11">Dave</text>

  <text x="500" y="102" text-anchor="middle" fill="#4e79a7" font-size="11">Alice</text>
  <rect x="555" y="88" width="40" height="22" rx="2" fill="#4e79a7"/><text x="575" y="103" text-anchor="middle" fill="white" font-size="11">1.0</text>
  <rect x="625" y="88" width="40" height="22" rx="2" fill="#c0d4e8"/><text x="645" y="103" text-anchor="middle" fill="#333" font-size="11">0.2</text>
  <rect x="695" y="88" width="40" height="22" rx="2" fill="#3a6ea5"/><text x="715" y="103" text-anchor="middle" fill="white" font-size="11">0.7</text>
  <rect x="765" y="88" width="40" height="22" rx="2" fill="#e8f0fb"/><text x="785" y="103" text-anchor="middle" fill="#333" font-size="11">-0.5</text>

  <text x="500" y="132" text-anchor="middle" fill="#333" font-size="11">Bob</text>
  <rect x="555" y="118" width="40" height="22" rx="2" fill="#c0d4e8"/><text x="575" y="133" text-anchor="middle" fill="#333" font-size="11">0.2</text>
  <rect x="625" y="118" width="40" height="22" rx="2" fill="#4e79a7"/><text x="645" y="133" text-anchor="middle" fill="white" font-size="11">1.0</text>
  <rect x="695" y="118" width="40" height="22" rx="2" fill="#94b8d0"/><text x="715" y="133" text-anchor="middle" fill="#333" font-size="11">0.4</text>
  <rect x="765" y="118" width="40" height="22" rx="2" fill="#e8f0fb"/><text x="785" y="133" text-anchor="middle" fill="#333" font-size="11">-0.3</text>

  <text x="500" y="162" text-anchor="middle" fill="#333" font-size="11">Carol</text>
  <rect x="555" y="148" width="40" height="22" rx="2" fill="#3a6ea5"/><text x="575" y="163" text-anchor="middle" fill="white" font-size="11">0.7</text>
  <rect x="625" y="148" width="40" height="22" rx="2" fill="#94b8d0"/><text x="645" y="163" text-anchor="middle" fill="#333" font-size="11">0.4</text>
  <rect x="695" y="148" width="40" height="22" rx="2" fill="#4e79a7"/><text x="715" y="163" text-anchor="middle" fill="white" font-size="11">1.0</text>
  <rect x="765" y="148" width="40" height="22" rx="2" fill="#f0d0d0"/><text x="785" y="163" text-anchor="middle" fill="#333" font-size="11">-0.8</text>

  <text x="500" y="192" text-anchor="middle" fill="#333" font-size="11">Dave</text>
  <rect x="555" y="178" width="40" height="22" rx="2" fill="#e8f0fb"/><text x="575" y="193" text-anchor="middle" fill="#333" font-size="11">-0.5</text>
  <rect x="625" y="178" width="40" height="22" rx="2" fill="#f0d0d0"/><text x="645" y="193" text-anchor="middle" fill="#333" font-size="11">-0.3</text>
  <rect x="695" y="178" width="40" height="22" rx="2" fill="#f0d0d0"/><text x="715" y="193" text-anchor="middle" fill="#333" font-size="11">-0.8</text>
  <rect x="765" y="178" width="40" height="22" rx="2" fill="#4e79a7"/><text x="785" y="193" text-anchor="middle" fill="white" font-size="11">1.0</text>

  <text x="610" y="242" text-anchor="middle" fill="#555" font-size="11">Alice and Carol most similar (0.7)</text>
  <text x="610" y="260" text-anchor="middle" fill="#555" font-size="11">Carol rated Item C = 5  →  predict Alice ≈ high</text>
</svg>
'''
display(SVG(svg))


## Derivation

### Why Pearson Correlation?

Raw cosine similarity ignores rating scale biases (a strict rater who gives 3 means "good"; a lenient rater who gives 3 means "bad"). Pearson correlation **mean-centres** each user's ratings:

$$r_{ui}^{\text{centred}} = r_{ui} - \bar{r}_u, \quad \bar{r}_u = \frac{1}{|I_u|} \sum_{i \in I_u} r_{ui}$$

This makes ratings comparable across users with different average tendencies.

### Pearson Similarity

$$\text{sim}(u, v) = \frac{\sum_{i \in I_u \cap I_v} (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_{i \in I_u \cap I_v} (r_{ui} - \bar{r}_u)^2} \cdot \sqrt{\sum_{i \in I_u \cap I_v} (r_{vi} - \bar{r}_v)^2}}$$

Sum only over items **co-rated** by both $u$ and $v$.

### Rating Prediction

Given $k$ nearest neighbours $\mathcal{N}_k(u)$ who have rated item $i$:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u,v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u,v)|}$$

The numerator is a weighted sum of *centred* ratings; the denominator normalises. Adding $\bar{r}_u$ back restores the original scale.

**Intuition:** If similar users liked item $i$ above their average, predict $u$ will too — by a commensurate amount.


## Algorithm Steps

1. **Compute user mean** $\bar{r}_u$ for each user $u$
2. **Compute Pearson similarity** $\text{sim}(u, v)$ for all user pairs $(u, v)$ sharing co-rated items
3. **For each (user $u$, unrated item $i$):**
   - Find neighbours $v$ who rated $i$, sorted by $|\text{sim}(u,v)|$ descending
   - Select top-$k$ neighbours
   - Compute $\hat{r}_{ui}$ using the weighted formula above
4. **Recommend** items with highest predicted rating


In [ ]:
import numpy as np
from collections import defaultdict


def pearson_similarity(u_ratings, v_ratings):
    """
    Compute Pearson correlation between users u and v
    over their co-rated items.

    u_ratings, v_ratings : dict {item: rating}
    Returns float in [-1, 1] or 0.0 if no co-rated items
    """
    common = set(u_ratings) & set(v_ratings)
    if len(common) < 2:
        return 0.0

    u_vals = np.array([u_ratings[i] for i in common])
    v_vals = np.array([v_ratings[i] for i in common])

    u_mean = u_vals.mean()
    v_mean = v_vals.mean()

    u_c = u_vals - u_mean
    v_c = v_vals - v_mean

    denom = np.linalg.norm(u_c) * np.linalg.norm(v_c)
    if denom == 0:
        return 0.0

    return float(np.dot(u_c, v_c) / denom)


def user_user_cf(ratings, target_user, target_item, k=3):
    """
    Predict the rating of target_user on target_item using user-user CF.

    Inputs
    ------
    ratings     : dict {user: {item: rating}}
    target_user : user to predict for
    target_item : item to predict
    k           : int — number of nearest neighbours to use

    Output
    ------
    float — predicted rating
    """
    target_ratings = ratings[target_user]
    target_mean = np.mean(list(target_ratings.values()))

    # Find k most similar users who have rated target_item
    sims = []
    for user, user_ratings in ratings.items():
        if user == target_user:
            continue
        if target_item not in user_ratings:
            continue
        sim = pearson_similarity(target_ratings, user_ratings)
        sims.append((sim, user))

    # Sort by absolute similarity (most similar first)
    sims.sort(key=lambda x: -abs(x[0]))
    neighbours = sims[:k]

    if not neighbours:
        return target_mean  # fallback to user mean

    # Weighted average of neighbours' mean-centred ratings
    numerator = 0.0
    denominator = 0.0
    for sim, user in neighbours:
        user_ratings = ratings[user]
        user_mean = np.mean(list(user_ratings.values()))
        numerator   += sim * (user_ratings[target_item] - user_mean)
        denominator += abs(sim)

    if denominator == 0:
        return target_mean

    return target_mean + numerator / denominator


# ── Demo ──────────────────────────────────────────────────────────────────────
ratings = {
    "Alice": {"A": 5, "B": 4, "D": 2},
    "Bob":   {"B": 3, "C": 4},
    "Carol": {"A": 4, "C": 5, "D": 3},
    "Dave":  {"A": 2, "B": 1, "D": 5},
}

# Compute similarity matrix
users = list(ratings.keys())
print("Pearson similarity matrix:")
print(f"{'':>8}", end="")
for v in users:
    print(f"{v:>8}", end="")
print()
for u in users:
    print(f"{u:>8}", end="")
    for v in users:
        sim = pearson_similarity(ratings[u], ratings[v])
        print(f"{sim:>8.2f}", end="")
    print()

# Predict Alice's rating for Item C
pred = user_user_cf(ratings, "Alice", "C", k=3)
print(f"\nPredicted rating for Alice on Item C: {pred:.2f}")
print(f"(Carol rated C=5 and is most similar to Alice)")
